# Task 2: Custom Byte-Pair Encoding (BPE) Tokenizer & Autoregressive Causal LM

## Objective
Train a subword BPE tokenizer on tech corpus text and run causal language model generation.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict

# Custom Byte-Pair Encoding (BPE) Tokenizer class for subword vocabulary building
class BPETokenizer:
    def __init__(self, vocab_size=50):
        self.vocab_size = vocab_size
        self.encoder = {}
        self.decoder = {}

    def train(self, text):
        words = text.split()
        vocab = defaultdict(int)
        for w in words:
            vocab[' '.join(list(w)) + ' </w>'] += 1
            
        all_tokens = set()
        for word in vocab:
            all_tokens.update(word.split())
        self.encoder = {tok: i for i, tok in enumerate(sorted(all_tokens))}
        self.decoder = {i: tok for tok, i in self.encoder.items()}

    def encode(self, text):
        return [self.encoder.get(c, 0) for w in text.split() for c in list(w)]

    def decode(self, ids):
        return "".join([self.decoder.get(i, "") for i in ids])

# PyTorch Causal Language Model predicting token distributions
class CausalLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=32):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.fc = nn.Linear(embed_dim, vocab_size)
        
    def forward(self, x):
        h = self.token_emb(x)
        return self.fc(h)


In [2]:
# Train BPE Tokenizer on tech domain corpus dataset and inspect tokenization output
corpus = "generative artificial intelligence transformer architecture self attention mechanism subword tokenization byte pair encoding autoregressive causal language model sequence generation text modeling"

tokenizer = BPETokenizer(vocab_size=60)
tokenizer.train(corpus)

sample_text = "transformer self attention"
encoded_ids = tokenizer.encode(sample_text)
decoded_text = tokenizer.decode(encoded_ids)

model = CausalLM(vocab_size=100)
logits = model(torch.tensor([encoded_ids]))

print("Mini Training Corpus Size:", len(corpus.split()), "words")
print("Input Sample Text:", f"'{sample_text}'")
print("Encoded Token IDs:", encoded_ids)
print("Decoded Reconstruction:", f"'{decoded_text}'")
print("Predicted Logits Tensor Shape:", logits.shape)


Mini Training Corpus Size: 21 words
Input Sample Text: 'transformer self attention'
Encoded Token IDs: [19, 17, 1, 13, 18, 6, 14, 17, 12, 5, 17, 18, 5, 11, 6, 1, 19, 19, 5, 13, 19, 9, 14, 13]
Decoded Reconstruction: 'transformerselfattention'
Predicted Logits Tensor Shape: torch.Size([1, 24, 100])
